In [ ]:
import pandas as pd

df = pd.read_csv("/media/mudda/extra/training_dataset.csv")
print("Before drop:", df.shape)


In [16]:
USELESS_COLUMNS = [
    "Unnamed: 0",

    # Metadata / identifiers not needed for ML
    "collection_type",
    "alloc_collection_id",
    "collection_events_type",
    "collection_name",
    "collection_logical_name",
    "start_after_collection_ids",
    "user",
    "cluster",
    "event",
    "instance_events_type",

    # Complex / nested structures (need feature extraction, not direct use)
    "resource_request",
    "constraint",
    "cpu_usage_distribution",
    "tail_cpu_usage_distribution"
]


In [17]:
df_clean = df.drop(columns=USELESS_COLUMNS, errors="ignore")
print("After drop:", df_clean.shape)


After drop: (405894, 20)


In [18]:
df_clean.columns.tolist()


['time',
 'collection_id',
 'scheduling_class',
 'priority',
 'instance_index',
 'machine_id',
 'collections_events_type',
 'vertical_scaling',
 'scheduler',
 'start_time',
 'end_time',
 'average_usage',
 'maximum_usage',
 'random_sample_usage',
 'assigned_memory',
 'page_cache_memory',
 'cycles_per_instruction',
 'memory_accesses_per_instruction',
 'sample_rate',
 'failed']

In [19]:
df_clean.to_csv("final_removed.csv", index=False)


In [7]:
import pandas as pd
import numpy as np
import ast

# -----------------------------
# 1. LOAD RAW DATA
# -----------------------------
df = pd.read_csv("final_removed.csv")

# -----------------------------
# 2. CREATE DURATION FEATURE
# -----------------------------
df["duration"] = df["end_time"] - df["start_time"]

# -----------------------------
# 3. SAFE DICTIONARY PARSER
# -----------------------------
def safe_extract(value, key):
    if pd.isna(value):
        return 0.0
    if isinstance(value, dict):
        return value.get(key, 0.0)
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, dict):
                return parsed.get(key, 0.0)
        except:
            return 0.0
    return 0.0

# Extract usage features
df["avg_cpu"] = df["average_usage"].apply(lambda x: safe_extract(x, "cpus"))
df["avg_memory"] = df["average_usage"].apply(lambda x: safe_extract(x, "memory"))

df["max_cpu"] = df["maximum_usage"].apply(lambda x: safe_extract(x, "cpus"))
df["max_memory"] = df["maximum_usage"].apply(lambda x: safe_extract(x, "memory"))

df["sample_cpu"] = df["random_sample_usage"].apply(lambda x: safe_extract(x, "cpus"))

# -----------------------------
# 4. DROP UNUSABLE COLUMNS
# -----------------------------
df.drop(columns=[
    "collection_id",
    "instance_index",
    "machine_id",
    "start_time",
    "end_time",
    "average_usage",
    "maximum_usage",
    "random_sample_usage"
], errors="ignore", inplace=True)

# -----------------------------
# 5. DROP CONSTANT COLUMNS
# -----------------------------
for col in df.columns:
    if df[col].nunique() <= 1:
        df.drop(columns=[col], inplace=True)

# -----------------------------
# 6. HANDLE NaN
# -----------------------------
df = df.fillna(0)

# -----------------------------
# 7. REMOVE DUPLICATES
# -----------------------------
df = df.drop_duplicates()

# -----------------------------
# 8. SORT BY TIME
# -----------------------------
df = df.sort_values("time")

print("Final shape:", df.shape)
print(df.head())


Final shape: (336038, 18)
       time  scheduling_class  priority  collections_events_type  \
0         0                 3       200                        2   
13149     0                 3       200                        2   
13235     0                 1       119                        2   
13387     0                 1       200                        0   
13439     0                 1       105                        2   

       vertical_scaling  scheduler  assigned_memory  page_cache_memory  \
0                   1.0        0.0         0.014435           0.000415   
13149               1.0        0.0         0.032593           0.000143   
13235               2.0        0.0         0.005211           0.000004   
13387               1.0        0.0         0.003906           0.000312   
13439               2.0        1.0         0.005241           0.000397   

       cycles_per_instruction  memory_accesses_per_instruction  sample_rate  \
0                    0.000000            

In [6]:
df.to_csv("ready_data.csv", index=False)
print("Dataset saved successfully.")


NameError: name 'df' is not defined

In [7]:
import pandas as pd

df_final = pd.read_csv("ready_data.csv")

print("Shape:", df_final.shape)

print("\nClass distribution:")
print(df_final["failed"].value_counts())
print("\nClass ratio:")
print(df_final["failed"].value_counts(normalize=True))


Shape: (336038, 18)

Class distribution:
failed
0    279665
1     56373
Name: count, dtype: int64

Class ratio:
failed
0    0.832242
1    0.167758
Name: proportion, dtype: float64


In [8]:
print("Duplicate rows:", df_final.duplicated().sum())
print("Total rows:", len(df_final))


Duplicate rows: 0
Total rows: 336038


In [9]:
print(df_final["failed"].value_counts())
print(df_final["failed"].value_counts(normalize=True))


failed
0    279665
1     56373
Name: count, dtype: int64
failed
0    0.832242
1    0.167758
Name: proportion, dtype: float64


In [11]:
print("Number of rows:",df_final.shape[0])


Number of rows: 336038


In [14]:
df_final.summary()


AttributeError: 'DataFrame' object has no attribute 'summary'